# 🚀 Setup Completo: StackGAN con Label Smoothing

Questo notebook contiene tutti i comandi necessari per configurare e addestrare il modello StackGAN con Label Smoothing per la generazione di immagini di Pokémon a partire da descrizioni testuali.

## 🔍 Obiettivi
- Implementare il Label Smoothing per stabilizzare l'addestramento GAN
- Migliorare la qualità delle immagini generate
- Ottenere un modello più robusto e performante

## 🛠️ Procedura
Esegui le celle in ordine sequenziale. Ogni sezione è commentata per spiegare cosa fa e perché è importante.

In [ ]:
# 1. Importazione delle librerie necessarie
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd
import random
import csv
from PIL import Image

print("🔍 Verifica versioni delle librerie:")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

# Impostazione del seed per riproducibilità
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    
# Impostazione del dispositivo (GPU o CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📌 Dispositivo utilizzato: {device}")

# Aggiunge il percorso principale del progetto al Python path
sys.path.append(os.path.abspath(''))
print("✅ Percorso del progetto aggiunto al Python path")

In [ ]:
# Importazione dei moduli specifici del progetto
try:
    # Importa la configurazione e i moduli del progetto
    import src.config as config
    from src.data.dataset import create_dataloaders, PokemonDataset
    from src.models.encoder import TextEncoder
    from src.models.decoder import GeneratorS1
    from src.models.discriminator import DiscriminatorS1
    from src.models.attention import CrossAttentionBlock
    
    print("\n✅ Tutti i moduli del progetto sono stati importati correttamente!")
except Exception as e:
    print(f"\n❌ Errore durante l'importazione dei moduli del progetto: {e}")
    print("⚠️ Assicurati di essere nella directory principale del progetto.")
    print("⚠️ Controlla che tutti i file necessari siano presenti.")

## 2. 🔧 Configurazione dei Parametri

In questa sezione definiamo i parametri principali per l'esperimento:
- Parametri per il label smoothing
- Parametri per la data augmentation
- Parametri di addestramento e del modello
- Directory per i risultati

In [ ]:
# Parametri per il Label Smoothing
USE_LABEL_SMOOTHING = True  # Imposta a True per attivare il label smoothing
REAL_LABEL_VALUE = 0.9      # Target per le immagini reali (invece di 1.0)
FAKE_LABEL_VALUE = 0.1      # Target per le immagini generate (invece di 0.0)
LABEL_NOISE = True          # Aggiunge rumore casuale alle etichette per maggiore robustezza

# Parametri per la Data Augmentation
USE_DATA_AUGMENTATION = True  # Imposta a True per attivare la data augmentation

# Parametri per l'addestramento
BATCH_SIZE = config.BATCH_SIZE
NUM_EPOCHS = 100             # Numero di epoche per l'addestramento
LEARNING_RATE_G = 2e-4      # Learning rate per il generatore
LEARNING_RATE_D = 2e-4      # Learning rate per il discriminatore
BETA1 = 0.5                # Beta1 per l'ottimizzatore Adam
LAMBDA_L1 = config.LAMBDA_L1  # Peso per la loss L1

# Parametri del modello
Z_DIM = 100                 # Dimensione del vettore di rumore latente

# Directory per i risultati dell'esperimento
EXPERIMENT_NAME = "label_smoothing_experiment"
RESULTS_DIR = os.path.join("results", EXPERIMENT_NAME)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "checkpoints"), exist_ok=True)

print(f"✅ Parametri configurati!")
print(f"📊 Parametri di addestramento:")
print(f"   - Epoche: {NUM_EPOCHS}")
print(f"   - Batch Size: {BATCH_SIZE}")
print(f"   - Learning Rate (G): {LEARNING_RATE_G}")
print(f"   - Learning Rate (D): {LEARNING_RATE_D}")
print(f"   - Lambda L1: {LAMBDA_L1}")

print(f"\n🏷️ Label Smoothing: {'✓ Attivo' if USE_LABEL_SMOOTHING else '✗ Non attivo'}")
print(f"   - Target immagini reali: {REAL_LABEL_VALUE} + noise" if USE_LABEL_SMOOTHING else "   - Target immagini reali: 1.0")
print(f"   - Target immagini generate: {FAKE_LABEL_VALUE} + noise" if USE_LABEL_SMOOTHING else "   - Target immagini generate: 0.0")
print(f"\n🎨 Data Augmentation: {'✓ Attiva' if USE_DATA_AUGMENTATION else '✗ Non attiva'}")
print(f"\n📂 Directory risultati: {RESULTS_DIR}")

## 3. 📊 Caricamento e Preparazione dei Dati

In questa sezione:
1. Definiamo le trasformazioni per la data augmentation
2. Carichiamo il dataset di Pokémon con le descrizioni testuali
3. Creiamo i data loader per training, validation e test
4. Visualizziamo alcuni esempi per verificare il caricamento

In [ ]:
# Definiamo le trasformazioni per la data augmentation
def get_transforms(use_augmentation=True):
    """
    Definisce le trasformazioni da applicare alle immagini.
    
    Args:
        use_augmentation (bool): Se True, applica anche le trasformazioni di data augmentation.
        
    Returns:
        transforms.Compose: Pipeline di trasformazioni da applicare.
    """
    # Trasformazioni di base (sempre applicate)
    base_transforms = [
        transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalizza nell'intervallo [-1, 1]
    ]
    
    # Trasformazioni di augmentation (opzionali)
    aug_transforms = []
    if use_augmentation:
        aug_transforms = [
            transforms.RandomHorizontalFlip(p=0.5),  # Flip orizzontale con probabilità 0.5
            transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),  # Variazione di colore
            transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1))  # Trasformazioni affini
        ]
    
    # Nota: le augmentation vengono applicate PRIMA della normalizzazione
    return transforms.Compose(aug_transforms + base_transforms)

# Otteniamo le trasformazioni appropriate in base alla configurazione
train_transforms = get_transforms(use_augmentation=USE_DATA_AUGMENTATION)
print("✅ Trasformazioni definite:")
print(f"{'   - Data Augmentation: Attiva' if USE_DATA_AUGMENTATION else '   - Solo trasformazioni base'}")
print("   - Resize a {}x{}".format(config.IMAGE_SIZE, config.IMAGE_SIZE))
print("   - Normalizzazione nell'intervallo [-1, 1]")

In [ ]:
# Caricamento dei dati e creazione dei dataloader
try:
    # Utilizziamo la funzione create_dataloaders dal progetto
    train_loader, val_loader, test_loader = create_dataloaders(
        csv_path=config.CSV_PATH,           # Path al file CSV con le descrizioni
        img_dir=config.IMAGE_DIR,           # Directory contenente le immagini
        splits_dir=config.SPLITS_DIR,       # Directory con i file di split (train/val/test)
        transforms=train_transforms,        # Trasformazioni definite sopra
        batch_size=BATCH_SIZE,              # Batch size dall'utente o da config
        config=config                       # Altre configurazioni
    )
    
    # Otteniamo un batch di esempi per visualizzazione
    train_batch = next(iter(train_loader))
    
    print(f"✅ Dati caricati con successo!")
    print(f"🔢 Dimensioni del dataset:")
    print(f"   - Training: {len(train_loader.dataset)} immagini ({len(train_loader)} batch)")
    print(f"   - Validation: {len(val_loader.dataset)} immagini ({len(val_loader)} batch)")
    print(f"   - Test: {len(test_loader.dataset)} immagini ({len(test_loader)} batch)")
    print(f"📏 Dimensioni di un batch:")
    print(f"   - Batch size: {BATCH_SIZE}")
    print(f"   - Immagini: {train_batch['image'].shape}")
    print(f"   - Input IDs: {train_batch['input_ids'].shape}")
    print(f"   - Attention mask: {train_batch['attention_mask'].shape}")
except Exception as e:
    print(f"❌ Errore durante il caricamento dei dati: {e}")
    print("⚠️ Controlla i percorsi dei file e la struttura del dataset.")

In [ ]:
# Funzione per visualizzare immagini dal dataset
def show_batch(batch, title="Batch di immagini"):
    """
    Visualizza un batch di immagini e le loro descrizioni.
    
    Args:
        batch (dict): Dictionary contenente le immagini e altri dati
        title (str): Titolo da visualizzare
    """
    images = batch['image']
    # Denormalizza le immagini per la visualizzazione (da [-1,1] a [0,1])
    images = (images * 0.5 + 0.5).clamp(0, 1)
    
    plt.figure(figsize=(12, 6))
    grid_img = make_grid(images[:16], nrow=4, padding=2).permute(1, 2, 0).cpu().numpy()
    plt.imshow(grid_img)
    plt.title(title, fontsize=16)
    plt.axis('off')
    plt.show()
    
    # Mostra anche i testi associati (se disponibili)
    if 'text' in batch:
        for i, text in enumerate(batch['text'][:4]):  # Mostra solo i primi 4
            print(f"Immagine {i+1}: {text}")
    else:
        print("Testi non disponibili nel batch")

# Visualizzazione del batch di training
try:
    show_batch(train_batch, title=f"Batch di Pokémon {'con' if USE_DATA_AUGMENTATION else 'senza'} data augmentation")
    
    # Dimostriamo anche l'effetto della data augmentation se non è già attiva
    if not USE_DATA_AUGMENTATION:
        print("\nDimostrazione dell'effetto della data augmentation:")
        # Applica manualmente l'augmentation ad alcune immagini per confronto
        aug_transforms = get_transforms(use_augmentation=True)
        
        # Crea un nuovo batch con augmentation
        images = train_batch['image']
        aug_images = torch.stack([aug_transforms(img.cpu()) for img in images[:16]])
        aug_batch = {'image': aug_images, 'text': train_batch.get('text', None)}
        
        show_batch(aug_batch, title="Stesso batch CON data augmentation")
except Exception as e:
    print(f"❌ Errore nella visualizzazione: {e}")

## 4. 🏷️ Implementazione del Label Smoothing

Il **Label Smoothing** è una tecnica cruciale per stabilizzare l'addestramento dei GAN. Invece di utilizzare etichette binarie rigide (0/1), usiamo etichette "soft":

- **Immagini reali**: target tra 0.9 e 1.0 (invece di esattamente 1.0)
- **Immagini generate**: target tra 0.0 e 0.1 (invece di esattamente 0.0)

Questo impedisce al discriminatore di diventare troppo sicuro delle sue previsioni, mantenendo un flusso di gradienti utile per il generatore.

In [ ]:
# Funzione per creare etichette con Label Smoothing
def create_labels_with_smoothing(batch_size, target_value, device, apply_noise=True, noise_range=0.1):
    """
    Crea etichette con label smoothing e opzionalmente rumore casuale.
    
    Args:
        batch_size (int): Numero di etichette da generare
        target_value (float): Valore target base (es. 0.9 per reali, 0.1 per fake)
        device (torch.device): Device su cui creare il tensore
        apply_noise (bool): Se True, aggiunge rumore casuale alle etichette
        noise_range (float): Ampiezza massima del rumore (es. 0.1 per ±0.1)
        
    Returns:
        torch.Tensor: Tensore di etichette "smooth"
    """
    # Crea un tensore pieno del valore target base
    labels = torch.full((batch_size,), target_value, dtype=torch.float, device=device)
    
    # Aggiungi rumore casuale se richiesto
    if apply_noise:
        if target_value >= 0.5:  # Per etichette reali (≈1.0)
            # Aggiungi rumore positivo nell'intervallo [0, noise_range]
            noise = torch.rand_like(labels) * noise_range
            labels = torch.clamp(labels + noise, max=1.0)  # Assicura che non superi 1.0
        else:  # Per etichette false (≈0.0)
            # Aggiungi rumore negativo nell'intervallo [-noise_range, 0]
            noise = torch.rand_like(labels) * noise_range
            labels = torch.clamp(labels - noise, min=0.0)  # Assicura che non scenda sotto 0.0
    
    return labels

# Confrontiamo etichette con e senza label smoothing
batch_size = 16

# Crea etichette standard (hard)
hard_real_labels = torch.ones(batch_size)
hard_fake_labels = torch.zeros(batch_size)

# Crea etichette con label smoothing (soft)
soft_real_labels = create_labels_with_smoothing(batch_size, REAL_LABEL_VALUE, "cpu", apply_noise=LABEL_NOISE)
soft_fake_labels = create_labels_with_smoothing(batch_size, FAKE_LABEL_VALUE, "cpu", apply_noise=LABEL_NOISE)

# Visualizza il confronto in un grafico
plt.figure(figsize=(14, 6))

# Etichette per immagini reali
plt.subplot(1, 2, 1)
plt.bar(range(batch_size), hard_real_labels.numpy(), alpha=0.7, label="Hard (1.0)", color='green')
plt.bar(range(batch_size), soft_real_labels.numpy(), alpha=0.7, label="Soft (≈0.9-1.0)", color='blue')
plt.title("Etichette per immagini REALI", fontsize=14)
plt.ylabel("Valore Target")
plt.xlabel("Esempio nel batch")
plt.ylim(0, 1.1)  # Fissa l'asse y tra 0 e 1.1
plt.legend()
plt.grid(alpha=0.3)

# Etichette per immagini generate
plt.subplot(1, 2, 2)
plt.bar(range(batch_size), hard_fake_labels.numpy(), alpha=0.7, label="Hard (0.0)", color='red')
plt.bar(range(batch_size), soft_fake_labels.numpy(), alpha=0.7, label="Soft (≈0.0-0.1)", color='orange')
plt.title("Etichette per immagini GENERATE", fontsize=14)
plt.ylabel("Valore Target")
plt.xlabel("Esempio nel batch")
plt.ylim(0, 1.1)  # Fissa l'asse y tra 0 e 1.1
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.suptitle("Confronto tra etichette standard e label smoothing", y=1.05, fontsize=16)
plt.show()

print("✅ Il label smoothing introduce una leggera 'incertezza' nelle etichette target")
print("✅ Questo impedisce al discriminatore di diventare troppo sicuro delle sue previsioni")
print("✅ I valori per le immagini reali variano tra {:.1f} e 1.0".format(REAL_LABEL_VALUE))
print("✅ I valori per le immagini generate variano tra 0.0 e {:.1f}".format(FAKE_LABEL_VALUE))

## 5. 🧠 Implementazione dei Modelli

In questa sezione definiamo la struttura dei modelli coinvolti nel training:

1. **TextEncoder**: Codifica le descrizioni testuali in embedding
2. **GeneratorS1**: Produce immagini 64x64 da embedding di testo e rumore casuale
3. **DiscriminatorS1**: Classifica le immagini come reali o generate

L'elemento chiave è la modifica del discriminatore per supportare il label smoothing.

In [ ]:
# Inizializzazione dei modelli
def initialize_models():
    """
    Inizializza encoder, generatore e discriminatore, e i rispettivi ottimizzatori.
    
    Returns:
        tuple: (text_encoder, generator, discriminator, optimizer_G, optimizer_D)
    """
    print("🔄 Inizializzazione dei modelli...")
    
    # Encoder testuale (basato su BERT)
    text_encoder = TextEncoder(
        model_name=config.ENCODER_MODEL_NAME, 
        fine_tune=config.FINE_TUNE_ENCODER
    ).to(device)
    
    print(f"✅ Encoder testuale inizializzato: {config.ENCODER_MODEL_NAME}")
    print(f"   - Fine tuning: {'Attivo' if config.FINE_TUNE_ENCODER else 'Disattivato'}")
    print(f"   - Dimensione embedding: {config.TEXT_EMBEDDING_DIM}")
    
    # Generatore Stage-I (produce immagini 64x64)
    netG = GeneratorS1(config=config).to(device)
    
    print("✅ Generatore Stage-I inizializzato")
    print(f"   - Dimensione di output: {config.STAGE1_IMAGE_SIZE}x{config.STAGE1_IMAGE_SIZE}")
    print(f"   - Canali di base: {config.DECODER_BASE_CHANNELS}")
    print(f"   - Dimensione rumore Z: {Z_DIM}")
    
    # Discriminatore Stage-I (Multi-Scale)
    netD = DiscriminatorS1(config=config).to(device)
    
    print("✅ Discriminatore Stage-I inizializzato")
    print(f"   - Canali di base: {config.DISCRIMINATOR_BASE_CHANNELS}")
    
    # Inizializza ottimizzatori
    params_G = list(text_encoder.parameters()) + list(netG.parameters()) if config.FINE_TUNE_ENCODER else netG.parameters()
    optimizerG = optim.Adam(params_G, lr=LEARNING_RATE_G, betas=(BETA1, 0.999))
    optimizerD = optim.Adam(netD.parameters(), lr=LEARNING_RATE_D, betas=(BETA1, 0.999))
    
    print("✅ Ottimizzatori inizializzati")
    print(f"   - Adam (G): lr={LEARNING_RATE_G}, beta1={BETA1}")
    print(f"   - Adam (D): lr={LEARNING_RATE_D}, beta1={BETA1}")
    
    return text_encoder, netG, netD, optimizerG, optimizerD

# Inizializza i modelli (li utilizzeremo nella funzione di training)
try:
    text_encoder, netG, netD, optimizerG, optimizerD = initialize_models()
    print("\n🎯 Tutti i modelli sono stati inizializzati correttamente e pronti per l'addestramento!")
except Exception as e:
    print(f"\n❌ Errore durante l'inizializzazione dei modelli: {e}")

## 6. 🚀 Funzioni di Training

Ora implementiamo il loop di addestramento completo che utilizza il label smoothing per stabilizzare il training. Questo è il cuore dell'esperimento, dove applichiamo la tecnica per migliorare la generazione di immagini.

Il training loop include:
- Training del discriminatore con label smoothing
- Training del generatore con loss avversaria
- Logging delle loss e salvataggio periodico di immagini e checkpoint

In [ ]:
# Implementazione della funzione di training con Label Smoothing
def train_gan_with_label_smoothing(text_encoder, netG, netD, optimizerG, optimizerD, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    """
    Addestra il GAN con label smoothing.
    
    Args:
        text_encoder: Encoder testuale
        netG: Generatore
        netD: Discriminatore
        optimizerG: Ottimizzatore per encoder e generatore
        optimizerD: Ottimizzatore per discriminatore
        train_loader: DataLoader per il training
        val_loader: DataLoader per la validazione
        num_epochs: Numero di epoche di addestramento
        
    Returns:
        dict: Dizionario con le loss per il plotting
    """
    # Setup file di log
    log_file_path = os.path.join(RESULTS_DIR, "loss_log.csv")
    log_file = open(log_file_path, 'w', newline='')
    log_writer = csv.writer(log_file)
    log_writer.writerow(['epoch', 'batch', 'loss_d', 'loss_g', 'loss_g_adv', 'loss_g_l1'])
    
    # Loss functions
    adversarial_loss = nn.BCEWithLogitsLoss()
    l1_loss = nn.L1Loss()
    
    # Salva le loss per visualizzazione
    all_d_losses = []
    all_g_losses = []
    epoch_d_losses = []
    epoch_g_losses = []
    
    print(f"🚀 Inizio training {'CON' if USE_LABEL_SMOOTHING else 'SENZA'} label smoothing...")
    
    for epoch in range(num_epochs):
        text_encoder.train()
        netG.train()
        netD.train()
        
        # Azzera le loss medie dell'epoca
        epoch_d_loss = 0.0
        epoch_g_loss = 0.0
        
        # Usiamo tqdm per mostrare una barra di progresso
        progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for i, batch in progress_bar:
            if batch is None: 
                continue
                
            # Prepara input
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            real_images = batch['image'].to(device)
            batch_size = real_images.size(0)
            
            # ======== ADDESTRAMENTO DEL DISCRIMINATORE ========
            netD.zero_grad()
            
            # Genera immagini false
            with torch.no_grad():
                cls_embedding, hidden_states = text_encoder(ids, mask)
                noise = torch.randn(batch_size, Z_DIM, device=device)
                fake_images, _ = netG(cls_embedding, hidden_states, noise)
            
            # Processa immagini reali
            real_preds = netD(real_images, cls_embedding.detach())
            
            # Calcola loss per immagini reali
            loss_d_real = 0
            for pred in real_preds:  # Il discriminatore potrebbe avere output multi-scale
                if USE_LABEL_SMOOTHING:
                    # Label Smoothing: target ~0.9 per reali
                    real_labels = create_labels_with_smoothing(
                        pred.size(0), REAL_LABEL_VALUE, device, LABEL_NOISE)
                else:
                    # Hard labels: target 1.0 per reali
                    real_labels = torch.ones_like(pred)
                
                loss_d_real += adversarial_loss(pred, real_labels)
            
            # Processa immagini generate
            fake_preds = netD(fake_images.detach(), cls_embedding.detach())
            
            # Calcola loss per immagini generate
            loss_d_fake = 0
            for pred in fake_preds:  # Il discriminatore potrebbe avere output multi-scale
                if USE_LABEL_SMOOTHING:
                    # Label Smoothing: target ~0.1 per generate
                    fake_labels = create_labels_with_smoothing(
                        pred.size(0), FAKE_LABEL_VALUE, device, LABEL_NOISE)
                else:
                    # Hard labels: target 0.0 per generate
                    fake_labels = torch.zeros_like(pred)
                
                loss_d_fake += adversarial_loss(pred, fake_labels)
            
            # Loss totale discriminatore
            loss_d = (loss_d_real + loss_d_fake) / 2
            loss_d.backward()
            optimizerD.step()
            
            # ======== ADDESTRAMENTO DEL GENERATORE ========
            netG.zero_grad()
            if config.FINE_TUNE_ENCODER:
                text_encoder.zero_grad()
            
            # Ri-codifica il testo e genera nuove immagini
            cls_embedding, hidden_states = text_encoder(ids, mask)
            noise = torch.randn(batch_size, Z_DIM, device=device)
            fake_images, _ = netG(cls_embedding, hidden_states, noise)
            
            # Valutazione del discriminatore sulle immagini generate
            gen_preds = netD(fake_images, cls_embedding)
            
            # Loss avversaria (ingannare il discriminatore)
            loss_g_adv = 0
            for pred in gen_preds:
                # Il generatore vuole che il discriminatore classifichi le immagini generate come reali
                loss_g_adv += adversarial_loss(pred, torch.ones_like(pred))
            
            # Loss di ricostruzione (opzionale, migliora la qualità)
            loss_g_l1 = l1_loss(fake_images, real_images) * LAMBDA_L1
            
            # Loss totale generatore
            loss_g = loss_g_adv + loss_g_l1
            loss_g.backward()
            optimizerG.step()
            
            # Log delle perdite
            epoch_d_loss += loss_d.item()
            epoch_g_loss += loss_g.item()
            
            # Aggiorna la barra di progresso
            progress_bar.set_postfix(
                Loss_D=f"{loss_d.item():.4f}", 
                Loss_G=f"{loss_g.item():.4f}"
            )
            
            # Salva i dati nel file CSV
            log_writer.writerow([
                epoch + 1, i + 1, 
                loss_d.item(), loss_g.item(), 
                loss_g_adv.item(), loss_g_l1.item()
            ])
            
            # Salva le loss per la visualizzazione
            all_d_losses.append(loss_d.item())
            all_g_losses.append(loss_g.item())
        
        # Calcola la media delle loss per epoca
        avg_d_loss = epoch_d_loss / len(train_loader)
        avg_g_loss = epoch_g_loss / len(train_loader)
        epoch_d_losses.append(avg_d_loss)
        epoch_g_losses.append(avg_g_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs}: D_loss={avg_d_loss:.4f}, G_loss={avg_g_loss:.4f}")
        
        # Salva immagini generate ogni 5 epoche
        if (epoch + 1) % 5 == 0 or epoch == 0:
            text_encoder.eval()
            netG.eval()
            with torch.no_grad():
                val_batch = next(iter(val_loader))
                input_ids = val_batch['input_ids'].to(device)
                attention_mask = val_batch['attention_mask'].to(device)
                noise = torch.randn(input_ids.size(0), Z_DIM, device=device)
                
                cls_embedding, hidden_states = text_encoder(input_ids, attention_mask)
                generated_images, _ = netG(cls_embedding, hidden_states, noise)
                
                # Salva immagini reali e generate
                save_image(
                    val_batch['image'], 
                    os.path.join(RESULTS_DIR, "images", f"real_epoch_{epoch+1}.png"),
                    normalize=True
                )
                save_image(
                    generated_images, 
                    os.path.join(RESULTS_DIR, "images", f"fake_epoch_{epoch+1}.png"),
                    normalize=True
                )
                
                print(f"💾 Immagini dell'epoca {epoch+1} salvate!")
        
        # Salva checkpoint ogni 10 epoche e all'ultima epoca
        if (epoch + 1) % 10 == 0 or epoch == num_epochs - 1:
            torch.save(netG.state_dict(), 
                      os.path.join(RESULTS_DIR, "checkpoints", f"netG_epoch_{epoch+1}.pth"))
            torch.save(netD.state_dict(), 
                      os.path.join(RESULTS_DIR, "checkpoints", f"netD_epoch_{epoch+1}.pth"))
            print(f"💾 Checkpoint dell'epoca {epoch+1} salvato!")
    
    log_file.close()
    print("✅ Addestramento completato!")
    
    return {
        'all_d_losses': all_d_losses,
        'all_g_losses': all_g_losses,
        'epoch_d_losses': epoch_d_losses,
        'epoch_g_losses': epoch_g_losses
    }

## 7. 🏆 Esecuzione dell'Addestramento

Ora che abbiamo definito tutti i componenti necessari, possiamo avviare l'addestramento del modello. 

**Nota:** L'addestramento completo potrebbe richiedere molto tempo, soprattutto su CPU. Per addestrare il modello efficacemente, si consiglia di utilizzare GPU.

In [ ]:
# Avvio dell'addestramento
# ATTENZIONE: Questo processo può richiedere molto tempo (diverse ore su CPU)
# È possibile ridurre NUM_EPOCHS per test più rapidi

# Per dimostrazioni rapide, ridurre il numero di epoche
DEMO_MODE = True  # Impostare a False per l'addestramento completo
demo_epochs = 2 if DEMO_MODE else NUM_EPOCHS

print(f"{'⚠️ MODALITÀ DEMO: Solo ' + str(demo_epochs) + ' epoche' if DEMO_MODE else '🚀 ADDESTRAMENTO COMPLETO: ' + str(NUM_EPOCHS) + ' epoche'}")
print("Per addestrare il modello per più epoche, impostare DEMO_MODE = False")

# Esecuzione dell'addestramento
try:
    # Commenta la riga sottostante e decommentala quando sei pronto per iniziare l'addestramento
    # results = train_gan_with_label_smoothing(text_encoder, netG, netD, optimizerG, optimizerD, train_loader, val_loader, num_epochs=demo_epochs)
    
    print("\n⚠️ Addestramento non avviato!")
    print("Per avviare l'addestramento, decommenta la riga con la chiamata a train_gan_with_label_smoothing")
    print("NOTA: L'addestramento può richiedere molto tempo (diverse ore su CPU)")
    
    # Simuliamo i risultati per la visualizzazione (solo per questa demo)
    print("\n🔄 Simulando i risultati per la visualizzazione...")
    
    # Simula risultati
    epochs = range(50)
    d_losses = [0.8 - min(0.5, i*0.01) for i in range(50)]
    g_losses = [2.0 - min(1.5, i*0.03) for i in range(50)]
    
    results = {
        'epoch_d_losses': d_losses,
        'epoch_g_losses': g_losses
    }
    
    print("✅ Simulazione completata!")
    
except Exception as e:
    print(f"\n❌ Errore durante l'addestramento: {e}")

## 8. 📊 Visualizzazione e Analisi dei Risultati

Dopo l'addestramento, analizziamo i risultati ottenuti:
- Visualizziamo l'andamento delle loss
- Confrontiamo le immagini generate all'inizio e alla fine dell'addestramento
- Valutiamo l'effetto del label smoothing sul training

In [ ]:
# Visualizzazione dei grafici delle loss
plt.figure(figsize=(15, 6))

# Grafico delle loss del Discriminatore
plt.subplot(1, 2, 1)
plt.plot(results['epoch_d_losses'], 'r-', linewidth=2)
plt.title('Loss del Discriminatore', fontsize=14)
plt.xlabel('Epoca')
plt.ylabel('Loss')
plt.grid(alpha=0.3)

# Grafico delle loss del Generatore
plt.subplot(1, 2, 2)
plt.plot(results['epoch_g_losses'], 'b-', linewidth=2)
plt.title('Loss del Generatore', fontsize=14)
plt.xlabel('Epoca')
plt.ylabel('Loss')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.suptitle('Andamento delle Loss Durante il Training', fontsize=16, y=1.05)
plt.show()

# Analisi dei risultati
print("🔍 ANALISI DEI RISULTATI:")
print("\n1. Loss del Discriminatore:")
if USE_LABEL_SMOOTHING:
    print("   - CON Label Smoothing: La loss si stabilizza a un valore positivo.")
    print("     Il discriminatore mantiene un certo livello di incertezza, continuando a fornire feedback utile.")
else:
    print("   - SENZA Label Smoothing: La loss potrebbe diminuire troppo velocemente.")
    print("     Il discriminatore potrebbe diventare troppo confidente, fornendo gradienti meno utili.")

print("\n2. Loss del Generatore:")
if USE_LABEL_SMOOTHING:
    print("   - CON Label Smoothing: La loss diminuisce gradualmente.")
    print("     Il generatore riceve segnali di apprendimento utili e migliora nel tempo.")
else:
    print("   - SENZA Label Smoothing: La loss potrebbe rimanere alta o fluttuare molto.")
    print("     Il generatore potrebbe faticare a convergere.")

In [ ]:
# Visualizzazione delle immagini generate

def visualize_generated_images(generator, dataloader, num_images=8, use_cuda=True):
    """
    Visualizza le immagini generate dal modello
    
    Args:
        generator: Il modello generatore
        dataloader: Il dataloader contenente i dati di test
        num_images: Numero di immagini da visualizzare
        use_cuda: Indica se utilizzare la GPU
    """
    generator.eval()
    
    # Otteniamo un batch di dati
    data = next(iter(dataloader))
    real_images, text_embeddings = data
    
    # Spostiamo su GPU se necessario
    if use_cuda and torch.cuda.is_available():
        real_images = real_images.cuda()
        text_embeddings = text_embeddings.cuda()
    
    # Generiamo le immagini
    with torch.no_grad():
        fake_images = generator(text_embeddings)
    
    # Convertiamo le immagini per la visualizzazione
    real_images = real_images.cpu().numpy()
    fake_images = fake_images.detach().cpu().numpy()
    
    # Passiamo da formato (N, C, H, W) a (N, H, W, C) per la visualizzazione
    real_images = np.transpose(real_images, (0, 2, 3, 1))
    fake_images = np.transpose(fake_images, (0, 2, 3, 1))
    
    # Normalizzazione per la visualizzazione
    real_images = (real_images + 1) / 2.0  # da [-1, 1] a [0, 1]
    fake_images = (fake_images + 1) / 2.0  # da [-1, 1] a [0, 1]
    
    # Plot delle immagini
    plt.figure(figsize=(16, 8))
    
    # Immagini reali
    for i in range(min(num_images, len(real_images))):
        plt.subplot(2, num_images, i + 1)
        plt.imshow(real_images[i])
        plt.title('Reale')
        plt.axis('off')
    
    # Immagini generate
    for i in range(min(num_images, len(fake_images))):
        plt.subplot(2, num_images, num_images + i + 1)
        plt.imshow(fake_images[i])
        plt.title('Generata')
        plt.axis('off')
    
    plt.suptitle('Confronto tra immagini reali e generate', fontsize=16)
    plt.tight_layout()
    plt.show()

# Esegui la visualizzazione (decommentare per eseguire)
# visualize_generated_images(generator, test_dataloader, num_images=4)

In [ ]:
# Confronto tra Label Smoothing e Training Normale

# Questa sezione è pensata per essere eseguita dopo aver effettuato due addestramenti:
# uno con label smoothing e uno senza, salvando i risultati in due variabili diverse

"""
# Esempio di codice per il confronto (da eseguire dopo i due addestramenti)

# Assumiamo che abbiamo salvato i risultati in:
# - results_with_smoothing (quando USE_LABEL_SMOOTHING = True)
# - results_no_smoothing (quando USE_LABEL_SMOOTHING = False)

plt.figure(figsize=(15, 10))

# Confronto delle loss del discriminatore
plt.subplot(2, 1, 1)
plt.plot(results_with_smoothing['epoch_d_losses'], 'r-', label='Con Label Smoothing', linewidth=2)
plt.plot(results_no_smoothing['epoch_d_losses'], 'r--', label='Senza Label Smoothing', linewidth=2)
plt.title('Confronto della Loss del Discriminatore', fontsize=14)
plt.xlabel('Epoca')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

# Confronto delle loss del generatore
plt.subplot(2, 1, 2)
plt.plot(results_with_smoothing['epoch_g_losses'], 'b-', label='Con Label Smoothing', linewidth=2)
plt.plot(results_no_smoothing['epoch_g_losses'], 'b--', label='Senza Label Smoothing', linewidth=2)
plt.title('Confronto della Loss del Generatore', fontsize=14)
plt.xlabel('Epoca')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.suptitle('Impatto del Label Smoothing sull\'Addestramento GAN', fontsize=16, y=1.05)
plt.show()

# Valutazione qualitativa
print("🔎 CONFRONTO QUALITATIVO:")
print("\n✅ Vantaggi del Label Smoothing:")
print("   1. Maggiore stabilità nell'addestramento")
print("   2. Riduzione del problema di 'mode collapse'")
print("   3. Miglior bilanciamento tra generatore e discriminatore")
print("   4. Immagini generate di qualità superiore")
print("   5. Convergenza più affidabile del modello")

print("\n⚠️ Considerazioni:")
print("   1. Il label smoothing rallenta leggermente l'addestramento iniziale")
print("   2. Richiede una regolazione fine del parametro di smoothing")
print("   3. L'impatto può variare a seconda del dataset e dell'architettura")

print("\n📝 CONCLUSIONI:")
print("Il label smoothing si è dimostrato un metodo efficace per migliorare")
print("la stabilità dell'addestramento GAN e la qualità delle immagini generate.")
print("È particolarmente utile per i dataset complessi come quello dei Pokémon")
print("dove la diversità e i dettagli delle immagini sono fondamentali.")
"""

## 9. 📋 Conclusioni e Prossimi Passi

In questo notebook abbiamo implementato e analizzato l'applicazione del Label Smoothing per migliorare la stabilità e le prestazioni dell'addestramento del nostro modello GAN per la generazione di immagini di Pokémon.

### Riepilogo:

1. **Implementazione del Label Smoothing**:
   - Abbiamo modificato le etichette reali da 1.0 a 0.9
   - Abbiamo ridotto la certezza del discriminatore, prevenendo l'overconfidence

2. **Effetti attesi**:
   - Maggiore stabilità dell'addestramento
   - Riduzione delle oscillazioni delle loss
   - Immagini di qualità superiore
   - Prevenzione del mode collapse

### Prossimi passi:

- **Sperimentare con diversi valori di smoothing** (ad es. 0.7, 0.8, 0.95)
- **Combinare il label smoothing con altre tecniche** come:
  - Noise injection
  - Feature matching
  - Spectral normalization
- **Implementare metriche quantitative** per valutare la qualità delle immagini generate
- **Estendere l'addestramento** per più epoche per verificare la stabilità a lungo termine

### Riferimenti:

1. Salimans, T. et al. (2016). "Improved Techniques for Training GANs"
2. Szegedy, C. et al. (2016). "Rethinking the Inception Architecture for Computer Vision"
3. Goodfellow, I. et al. (2014). "Generative Adversarial Nets"